# El Niño / La Niña Machine Learning Analysis

**Objective:** Use historical NOAA Oceanic Niño Index data to classify ENSO conditions.

This notebook avoids using the current ONI value as the sole predictor of a class that was itself defined from that same current ONI value. Instead, lagged ONI features are used to reduce direct target leakage.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)


## Load the dataset

If you are running the notebook from the GitHub repository in Colab, upload `ENSO_NOAA_ONI_1950_2026.csv` when prompted.

The file is stored in this repository under:

`04_El_Nino_La_Nina/data/ENSO_NOAA_ONI_1950_2026.csv`


In [ ]:
from google.colab import files

uploaded = files.upload()
filename = next(iter(uploaded))

df = pd.read_csv(filename)
df.head()


In [ ]:
print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nMissing values:\n", df.isnull().sum())
print("\nClass counts:\n", df["ENSO_Class"].value_counts())


In [ ]:
plt.figure(figsize=(8, 5))
df["ENSO_Class"].value_counts().plot(kind="bar")
plt.title("ENSO Class Distribution")
plt.xlabel("ENSO Class")
plt.ylabel("Number of Seasonal Observations")
plt.show()


## Feature selection

We use lagged ONI values and season number as predictors. We intentionally exclude the current `ONI` column from `X`, because `ENSO_Class` was generated from that same current ONI value.


In [ ]:
model_df = df.dropna(subset=[
    "ONI_Lag1",
    "ONI_Lag2",
    "ONI_Lag3",
    "ONI_Rolling3_Previous"
]).copy()

features = [
    "Season_Number",
    "ONI_Lag1",
    "ONI_Lag2",
    "ONI_Lag3",
    "ONI_Rolling3_Previous"
]

X = model_df[features]
y = model_df["ENSO_Class"]

X.head(), y.head()


## Time-aware split

Because ENSO observations are chronological, this notebook uses an earlier-period training set and a later-period test set rather than randomly mixing all years.


In [ ]:
split_year = 2015

train_mask = model_df["Year"] < split_year
test_mask = model_df["Year"] >= split_year

X_train = model_df.loc[train_mask, features]
y_train = model_df.loc[train_mask, "ENSO_Class"]

X_test = model_df.loc[test_mask, features]
y_test = model_df.loc[test_mask, "ENSO_Class"]

print("Training rows:", len(X_train))
print("Testing rows :", len(X_test))
print("Train years  :", model_df.loc[train_mask, "Year"].min(), "-", model_df.loc[train_mask, "Year"].max())
print("Test years   :", model_df.loc[test_mask, "Year"].min(), "-", model_df.loc[test_mask, "Year"].max())


In [ ]:
model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)
predictions = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, predictions))
print("\nClassification Report:\n")
print(classification_report(y_test, predictions))


In [ ]:
cm = confusion_matrix(y_test, predictions, labels=model.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)
disp.plot()
plt.title("ENSO Confusion Matrix")
plt.show()


In [ ]:
importance = pd.DataFrame({
    "Feature": features,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False)

importance


In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(importance["Feature"], importance["Importance"])
plt.title("Feature Importance")
plt.ylabel("Importance")
plt.xticks(rotation=25)
plt.show()


## Interpretation

- Lagged ONI values carry information about ENSO persistence.
- A time-aware split is more realistic than randomly mixing past and future observations.
- Accuracy alone is not enough; inspect precision, recall, F1-score and the confusion matrix.
- This is a classroom ML implementation, not an operational climate-forecasting system.
